## 🗺️ Lab Map: The Bike Demand Forecasting Flow

Below is the end-to-end journey for this lab. Each step represents a module in this workshop, moving from raw data to a monitored production model:

**Explore** $\xrightarrow{}$ **Prepare** $\xrightarrow{}$ **Train** $\xrightarrow{}$ **Register** $\xrightarrow{}$ **Serve** $\xrightarrow{}$ **Monitor**

--- 
*Current Module: **Monitor** (Module 06)*

# 📊 Module 6: Generate Reports for Data and Model Drift

### ✅ You're done when...
- A Regression Performance report is generated and saved as HTML.
- A Data Drift report is created, showing feature distribution changes.
- A Target Drift report is generated, identifying shifts in the target variable distribution.

In [ ]:
# Install requirements
!pip install -r ../requirements.txt

## 📦 Import Required Libraries

Before we proceed with training and tracking our machine learning model, we need to import the necessary libraries.


In [ ]:
# Import necessary modules
import os
import requests

import pandas as pd

from evidently import ColumnMapping
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, TargetDriftPreset, RegressionPreset
from evidently.metric_preset import DataQualityPreset
from evidently.metric_preset import RegressionPreset


## ✅ Set Necessary Variables

In [ ]:
# TODO: YOU SHOULD SET THE REQUIRED VARIABLES HERE

# Name of the model API Service or Route
# Similar to SERVICE_NAME.PROJECT_NAME.svc.cluster.local 
MODEL_API_SERVICE = ___ # Inference Endpoint

## 📁 Load Reference and Current Data

In [ ]:
# Load reference (January) and current (February) datasets
data_path = "./data/processed/"

# Read both CSV files
# TODO: YOU SHOULD SET THE PATH TO CLEAND DATASET FOR THE FIRST TWO MONTH
DATA_MONTH_1 = '___' # name of the file for first month
DATA_MONTH_2 = '___' # name of the file for second month
DATA_MONTH_3 = '___' # name of the file for third month
data_01 = pd.read_csv(data_path + f'{DATA_MONTH_1}')
data_02 = pd.read_csv(data_path + f'{DATA_MONTH_2}')
data_03 = pd.read_csv(data_path + f'{DATA_MONTH_3}')

reference_data = pd.concat([data_01, data_02, data_03], ignore_index=True)

# Load current dataset (March) 
# TODO: YOU SHOULD SET THE PATH TO CLEAND DATASET FOR CURRENT DATA_SOURCE (for example fourth MONTH)
current_data_source = '___' # name of the file for fourth month
current_data = pd.read_csv(data_path + f"{current_data_source}")

# Preview shapes and basic info
print("Reference data shape:", reference_data.shape)
print("Current data shape:", current_data.shape)
# reference_data.head()

## 📦 Add Predictions on Both Datasets as another Column 

We need them in the next step to generate the monitoring reports.

### Get prediciton on Reference Data

In [ ]:
# Define the subset of features expected by the model
model_features = [
    'temp', 'atemp', 'humidity', 'windspeed',
    'hour', 'weekday', 'season', 'holiday', 'workingday', 'weathersit'
]

# Extract the relevant subset and convert to JSON-like list of dicts
json_input_list_reference = reference_data[model_features].to_dict(orient='records')

# print(json_input_list_reference[:5])

In [ ]:
# ── Endpoint ─────────────────────────────────────────────────────────────
endpoint = f"http://{MODEL_API_SERVICE}:80/predict"

# ── Read the JSON-line file and send requests ────────────────────────────
ref_predictions = []

for item in json_input_list_reference:    
    inference_request = item
    response = requests.post(endpoint, json=inference_request)
    
    if response.status_code == 200:
        # FastAPI returns {"prediction": <value>}
        ref_predictions.append(response.json()["prediction"])
    
    else:
        print(f"Request failed with status code: {response.status_code}")
        print(f"Response content: {response.text}")

# ── Assemble results into a DataFrame ────────────────────────────────────
ref_pred_df = pd.DataFrame(ref_predictions, columns=["Predicted Count"])

# print(ref_pred_df.head())

In [ ]:
reference_data["prediction"] = ref_pred_df
reference_data.head()

### Get prediciton on Current Data

In [ ]:
# Define the subset of features expected by the model
model_features = [
    'temp', 'atemp', 'humidity', 'windspeed',
    'hour', 'weekday', 'season', 'holiday', 'workingday', 'weathersit'
]

# Extract the relevant subset and convert to JSON-like list of dicts
json_input_list_current = current_data[model_features].to_dict(orient='records')

# print(json_input_list_current[:5])

In [ ]:
# ── Endpoint ─────────────────────────────────────────────────────────────
endpoint = f"http://{MODEL_API_SERVICE}:80/predict"

# ── Read the JSON-line file and send requests ────────────────────────────
cur_predictions = []

for item in json_input_list_current:    
    inference_request = item
    response = requests.post(endpoint, json=inference_request)
    
    if response.status_code == 200:
        # FastAPI returns {"prediction": <value>}
        cur_predictions.append(response.json()["prediction"])
    
    else:
        print(f"Request failed with status code: {response.status_code}")
        print(f"Response content: {response.text}")

# ── Assemble results into a DataFrame ────────────────────────────────────
cur_pred_df = pd.DataFrame(cur_predictions, columns=["Predicted Count"])

# print(cur_pred_df.head())

In [ ]:
current_data["prediction"] = cur_pred_df
current_data.head()

## 🗺️ Define Column Mapping

Evidently supports specifying column roles explicitly via `ColumnMapping`, which helps produce more accurate and meaningful metrics.

Here, we define:
- `target`: the actual value to predict (`count`)
- `prediction`: (optional) placeholder for model prediction column
- `numerical_features`: continuous input features
- `categorical_features`: categorical or discrete input features


In [ ]:
target="count"
prediction="prediction"
numerical_features=['temp', 'atemp', 'humidity', 'windspeed', 'hour', 'weekday']
categorical_features=['holiday', 'workingday', 'weathersit']

# For now, we ignore season for data drift report,
# since it does not affect the conclusion
# categorical_features=['season', 'holiday', 'workingday', 'weathersit']

column_mapping = ColumnMapping(
    target="count",
    prediction="prediction",
    numerical_features=['temp', 'atemp', 'humidity', 'windspeed', 'hour', 'weekday'],
    categorical_features=['holiday', 'workingday', 'weathersit']
    )
# column_mapping.target = target
# column_mapping.prediction = prediction
# column_mapping.numerical_features = numerical_features
# column_mapping.categorical_features = categorical_features

## 📈 Generate a Regression Performance Report

The **Regression Performance Report** evaluates how well a model performs over time.

To simulate production monitoring, we'll assume that a `prediction` column already exists in the dataset (this could be added via an inference pipeline). The report will compare the predicted and actual target values (`count`) and show metrics like:
- RMSE
- R²
- Error distribution
- Prediction quality


In [ ]:
# Create the Regression Performance report
regression_report = Report(metrics=[RegressionPreset()])

# Hint: Use the RegressionPreset from evidently.metric_preset
regression_report.run(
    reference_data=reference_data,
    current_data=current_data,
    column_mapping=column_mapping
)

# Save the report as HTML
os.makedirs("./reports", exist_ok=True)
os.makedirs(f"./reports/{current_data_source}", exist_ok=True)
output_path = f"./reports/{current_data_source}/regression_performance_report.html"
regression_report.save_html(output_path)

print(f"✅ Regression Performance report saved to {output_path}")

## 📉 Generate a Data Drift Report

We'll use the `DataDriftReport` class from Evidently to compare feature distributions between the reference (Jan-Mar) and current (April) datasets.

This report will help us understand whether any input features have changed significantly, which may impact model predictions.


In [ ]:
# Create a report with the Data Drift preset
data_drift_report = Report(metrics=[DataDriftPreset()])

# Hint: Use the DataDriftPreset from evidently.metric_preset
data_drift_report.run(
        reference_data=reference_data[numerical_features + categorical_features], 
        current_data=current_data[numerical_features + categorical_features], 
        column_mapping=column_mapping
    )

# Create directories if they don't exist
# report_dir = "./reports"
# os.makedirs(report_dir, exist_ok=True)

# Save the report as HTML
output_path = f"./reports/{current_data_source}/data_drift_report.html"
data_drift_report.save_html(output_path)

print(f"✅ Data Drift report saved to {output_path}")

## 🎯 Generate a Target Drift Report

We'll now generate a **Target Drift Report** using Evidently.

This report focuses specifically on changes in the distribution of the **target variable** (`cnt`), which represents the total number of bike rentals. Drift in the target distribution can indicate seasonal or behavioral changes in users that may affect model performance.


In [ ]:
# Create the Target Drift report
target_drift_report = Report(metrics=[TargetDriftPreset()])

# Hint: Use the TargetDriftPreset from evidently.metric_preset
target_drift_report.run(
    reference_data=reference_data,
    current_data=current_data,
    column_mapping=column_mapping
)

# Save the report as HTML
output_path = f"./reports/{current_data_source}/target_drift_report.html"
target_drift_report.save_html(output_path)

print(f"✅ Target Drift report saved to {output_path}")